# Cruzamento de Leads Aulas Semanais

### Objeto: Separar nome, email e telefone dos leads que não estão nos grupos novos das Aulas Semanais

In [1]:
# Importações básicas
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

# Adiciona src ao path
sys.path.append('../src')

# Utilitários de dados
from data_utils import (
    load_raw_data,
    save_processed_data,
    remove_duplicates,
    handle_missing_values,
    detect_outliers,
    normalize_column,
    process_phone_string,
    process_phone_number,
    clean_and_lower_column,
    flatten_list_to_df,
    parse_utm_string
)

# Utilitários SQL
from sql_utils import DatabaseConnection as Dbc, load_query_from_file

# Utilitários de visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Utilitários de API
from api_utils import (
    make_request,
    get_json,
    post_json,
    paginated_request,
    response_to_dataframe
)

# utilitários hotmart
from hotmart_utils import Hotmart

# utilitários tmb
from tmb_utils import TMB   

# Configurações pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Load Database Driver
db = Dbc()

# Inicializar API Hotmart
hotmart = Hotmart()

# Inicializar API TMB
tmb = TMB()

print('✓ Importações concluídas com sucesso!')

✓ Importações concluídas com sucesso!


## Baixando Leads e Dataframe de Entradas

In [2]:
df_raw_leads = db.execute_query_from_file('select_complete_lead_data_from_campaign', {'conversion_type_id': '1', 'campaign_id': 'isca-aulassemanais'})

df_raw_leads['phone'] = process_phone_string(df_raw_leads, 'phone')
df_raw_leads['whatsapp_format'] = process_phone_string(df_raw_leads, 'whatsapp_format')


df_group_leads = load_raw_data('groups.csv')
df_group_leads['phone'] = process_phone_string(df_group_leads, 'Número')

In [5]:
leads_outside = df_raw_leads[
    ~(
        df_raw_leads['phone'].astype(str).isin(df_group_leads['phone'].astype(str)) |
        df_raw_leads['whatsapp_format'].astype(str).isin(df_group_leads['phone'].astype(str))
    )
]